# Market Making — Phase 1 Backtest

30-day simulation at `dt = 0.5 s` (~5 184 000 steps).  
Produces the full Task 7 backtesting report.

In [1]:
import sys, copy, time, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append('../src')

import utils.stock_simulation as stock_mod
import utils.market_simulator as market_mod
import utils.order_book.order_book_impl as book_mod
import utils.market_maker.quoter as quoter_mod
import utils.report.pnl_tracker as pnl_mod
import utils.report.controller as ctrl_mod
import utils.client_flow.flow_generator as flow_mod

for mod in [stock_mod, market_mod, book_mod, quoter_mod, pnl_mod, ctrl_mod, flow_mod]:
    importlib.reload(mod)

from utils.stock_simulation import Stock
from utils.market_simulator.market import Market
from utils.order_book.order_book_impl import Order_book
from utils.market_maker.quoter import Quoter, QuoterConfig
from utils.report.pnl_tracker import PnLTracker
from utils.report.controller import Controller
from utils.client_flow.flow_generator import ClientFlowGenerator

SEED    = 42
CAPITAL = 1_000_000.0
N_DAYS  = 1
DT_SEC  = 0.05   # 0.05 s steps — matches Phase 1 resolution
WINDOW  = 7_200   # 1-hour rolling window at 0.05 s steps

np.random.seed(SEED)
print('Modules loaded.')

Modules loaded.


## Build markets and controller

In [2]:
# ── Stock price path ─────────────────────────────────────────────────────────
stock = Stock(drift=0.0, vol=0.20)
stock.simulate_garch(n_days=N_DAYS, dt_seconds=DT_SEC,
                     alpha=0.05, beta=0.94, lam=756, sigma_J=0.005)
print(f'Stock: {stock.n_steps:,} steps, dt={stock.time_step:.1f}s, {N_DAYS} days')

# ── Reference markets B and C ────────────────────────────────────────────────
mkt_B = Market(stock=stock)
mkt_B.generate_noised_mid_price()
mkt_B.build_spread(option='Skew', window_size=WINDOW, alpha=0.5,
                   gamma=0.3, ema_span=5_000, threshold=3)
mkt_B.generate_depth(mean_eur=500_000)

mkt_C = copy.deepcopy(mkt_B)
mkt_C.build_spread(option='Adaptive', window_size=WINDOW)
mkt_C.generate_depth(mean_eur=200_000)

# ── Quoter config ────────────────────────────────────────────────────────────
cfg = QuoterConfig(
    requote_threshold_spread_fraction=0.25,
    delta_limit=0.90,
    hedge_partial_limit=0.80,
    emergency_penalty_multiplier=5.0,
)

# ── Order book, market maker, client flow ────────────────────────────────────
book = Order_book()
mm   = Quoter(mkt_B, mkt_C, config=cfg, capital_K=CAPITAL)
book.register_quoter_listener(mm.on_fill)

gen = ClientFlowGenerator(seed=SEED)
client_fn = lambda step, t, mid, bid, ask, dt: \
    gen.generate_step(mid_price=mid, best_bid=bid, best_ask=ask, dt=dt)

# ── Controller ───────────────────────────────────────────────────────────────
ctrl = Controller(mkt_B, mkt_C, book, mm, client_fn)
print('Controller ready.')

Stock: 1,728,000 steps, dt=0.1s, 1 days
Coefficient annualization : 20867.582514512793
Controller ready.


## Run simulation

In [3]:
t0 = time.time()
ctrl.simulate()
elapsed = time.time() - t0

df_trades = ctrl.trade_history
mm_fills  = df_trades[~df_trades['is_hedge']]
hedges    = df_trades[df_trades['is_hedge']]

print(f'Done in {elapsed:.1f}s')
print(f'Steps       : {len(ctrl.step_log):,}')
print(f'MM fills    : {len(mm_fills):,}  ({len(mm_fills)/N_DAYS:.0f}/day)')
print(f'Hedge legs  : {len(hedges):,}')
print(f'Inventory   : {mm.inventory:,.0f} EUR')
print(f'Quotes sent : {ctrl._n_quotes_posted:,}')

100%|██████████| 1728000/1728000 [03:04<00:00, 9371.11it/s] 


Done in 184.5s
Steps       : 1,728,000
MM fills    : 509  (509/day)
Hedge legs  : 170
Inventory   : 1,651 EUR
Quotes sent : 630,483


## Sanity checks — P&L identities

In [4]:
current_mid = ctrl._current_fair_mid()
rep = PnLTracker.report(df_trades, current_mid)

# Identity 1: realized + unrealized == total_mtm
d1 = abs(rep['realized_pnl'] + rep['unrealized_pnl'] - rep['total_mtm_pnl'])
assert d1 < 1e-4, f'Identity 1 broken: {d1}'
print(f'[OK] realized + unrealized = total_mtm  (Δ={d1:.2e})')

# Identity 2: inception + revaluation - fees == total_mtm
d2 = abs(rep['inception_spread_pnl'] + rep['inventory_revaluation_pnl']
         - rep['total_fees'] - rep['total_mtm_pnl'])
assert d2 < 1e-4, f'Identity 2 broken: {d2}'
print(f'[OK] inception + revaluation - fees = total_mtm  (Δ={d2:.2e})')

[OK] realized + unrealized = total_mtm  (Δ=0.00e+00)
[OK] inception + revaluation - fees = total_mtm  (Δ=0.00e+00)


## Backtesting report — Task 7

Prints the P&L summary and renders all required charts:
1. Best bid / ask / mid for the three markets
2. Bid / ask / mid evolution with top 10 trades by size
3. EUR/USD price alongside inventory over time
4. Average, median, 5th and 95th percentile MtM P&L since trade inception
5. Fill-rate analysis

In [5]:
ctrl.report()

════════════════════════════════════════════════════════════════════
  BACKTESTING REPORT — Phase 1   (1.0 days, dt=0.1s, 1,728,000 steps)
════════════════════════════════════════════════════════════════════
  Total MtM P&L                             -9254.93  USD
    Realized cash P&L                     -171110.57  USD
    Unrealized (open inventory × mid)      +161855.64  USD
────────────────────────────────────────────────────────────────────
  Inception spread P&L                     +39272.10  USD
    (spread captured at fill time)  
  Inventory revaluation P&L                -12236.91  USD
    (mid drift on open EUR position)
  Total fees paid                           36290.12  USD
    Maker fees (exchange A)                 14417.15  USD
    Taker fees (hedge B/C)                  21872.97  USD
────────────────────────────────────────────────────────────────────
  MM fills                                       509  (509/day)
  Hedge legs                                     17

AttributeError: 'QuoterConfig' object has no attribute 'stale_steps'